# Training & Tuning Random Forest (Hybrid LSTM + RF) di Google Colab

Notebook ini khusus untuk eksperimen **Tim Random Forest**:
1. Memuat `lstm_embeddings.npz` (hasil ekstraksi Tim LSTM) tanpa perlu melatih ulang LSTM.
2. Menggabungkan 32 fitur temporal LSTM + 9 fitur statis tanah/lingkungan.
3. Melakukan *hyperparameter tuning* dengan `TimeSeriesSplit` (anti-*data leakage*).
4. Membandingkan **Baseline RF** vs **Hybrid LSTM + RF** (`Accuracy`, `Precision`, `Recall`, `F1-Score`, `Top-3 Accuracy`).
5. Visualisasi *Confusion Matrix*, *Feature Importance*, dan ekspor model terbaik (`hybrid_rf_tuned.pkl`).

In [ ]:
# 1. Setup & Cek Library
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder

SEED = 42
np.random.seed(SEED)
print('Library siap!')

In [ ]:
# 2. Muat Data `lstm_embeddings.npz`
# Opsi A: Jika dijalankan di Google Colab dan ingin upload manual file lstm_embeddings.npz:
EMB_PATH = 'lstm_embeddings.npz'

if not os.path.exists(EMB_PATH):
    if os.path.exists('data/processed/lstm_embeddings.npz'):
        EMB_PATH = 'data/processed/lstm_embeddings.npz'
    else:
        try:
            from google.colab import files
            print('Silakan upload file lstm_embeddings.npz...')
            uploaded = files.upload()
            EMB_PATH = list(uploaded.keys())[0]
        except ImportError:
            raise FileNotFoundError('File lstm_embeddings.npz tidak ditemukan. Jalankan train_lstm.py dulu.')

bundle = np.load(EMB_PATH, allow_pickle=True)
emb_train = bundle['emb_train']
emb_test = bundle['emb_test']
X_seq_train = bundle['X_seq_train']
X_seq_test = bundle['X_seq_test']
X_stat_train = bundle['X_stat_train']
X_stat_test = bundle['X_stat_test']
y_train = bundle['y_train']
y_test = bundle['y_test']
classes = bundle['classes']

label_encoder = LabelEncoder()
label_encoder.classes_ = classes

STATIC_FEATURES = [
    'tekstur_tanah', 'ph_tanah', 'nitrogen', 'fosfor', 'kalium',
    'karbon_organik', 'drainase', 'ketinggian_tempat', 'risiko_bencana_hama'
]
HYBRID_FEATURES = [f'fitur_temporal_lstm_{i}' for i in range(emb_train.shape[1])] + STATIC_FEATURES

X_hybrid_train = np.hstack([emb_train, X_stat_train])
X_hybrid_test = np.hstack([emb_test, X_stat_test])
X_base_train = np.hstack([X_seq_train.mean(axis=1), X_stat_train])
X_base_test = np.hstack([X_seq_test.mean(axis=1), X_stat_test])

print(f'Train Hybrid: {X_hybrid_train.shape}, Test Hybrid: {X_hybrid_test.shape}, Kelas: {len(classes)}')

In [ ]:
# 3. Fungsi Evaluasi Lengkap (termasuk Top-K Accuracy)
def evaluate_model(name, clf, X_te, y_te, top_k=3):
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)
    k = min(top_k, len(clf.classes_))
    top_k_preds = clf.classes_[np.argsort(y_prob, axis=1)[:, -k:]]
    top_k_acc = np.mean([y_te[i] in top_k_preds[i] for i in range(len(y_te))])
    return {
        'Model': name,
        'Accuracy': round(float(accuracy_score(y_te, y_pred)), 4),
        'Precision (Weighted)': round(float(precision_score(y_te, y_pred, average='weighted', zero_division=0)), 4),
        'Recall (Weighted)': round(float(recall_score(y_te, y_pred, average='weighted', zero_division=0)), 4),
        'F1 (Weighted)': round(float(f1_score(y_te, y_pred, average='weighted', zero_division=0)), 4),
        f'Top-{k} Accuracy': round(float(top_k_acc), 4),
    }

In [ ]:
# 4. Latih Baseline & Hyperparameter Tuning Hybrid Random Forest
baseline_rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, class_weight='balanced', random_state=SEED, n_jobs=-1
)
baseline_rf.fit(X_base_train, y_train)

baseline_gb = GradientBoostingClassifier(random_state=SEED)
baseline_gb.fit(X_base_train, y_train)

hybrid_default_rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, class_weight='balanced', random_state=SEED, n_jobs=-1
)
hybrid_default_rf.fit(X_hybrid_train, y_train)

# Hyperparameter Tuning dengan TimeSeriesSplit (tanpa data leakage)
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
    'class_weight': [None, 'balanced', 'balanced_subsample'],
}

tscv = TimeSeriesSplit(n_splits=5)
search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=25,
    scoring='f1_weighted',
    cv=tscv,
    random_state=SEED,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_hybrid_train, y_train)
hybrid_tuned_rf = search.best_estimator_
print('\nBest Hyperparameters:', search.best_params_)

In [ ]:
# 5. Tabel Perbandingan Evaluasi
results = [
    evaluate_model('Baseline RF (Statis + Mean Iklim)', baseline_rf, X_base_test, y_test),
    evaluate_model('Baseline GradientBoosting', baseline_gb, X_base_test, y_test),
    evaluate_model('Hybrid LSTM + RF (Default)', hybrid_default_rf, X_hybrid_test, y_test),
    evaluate_model('Hybrid LSTM + RF (Tuned)', hybrid_tuned_rf, X_hybrid_test, y_test),
]
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# 6. Visualisasi Confusion Matrix & Top-15 Feature Importance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

y_pred_tuned = hybrid_tuned_rf.predict(X_hybrid_test)
cm = confusion_matrix(y_test, y_pred_tuned, labels=hybrid_tuned_rf.classes_)
class_names = label_encoder.inverse_transform(hybrid_tuned_rf.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix - Hybrid LSTM + RF (Tuned)')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

imp_df = pd.DataFrame({
    'fitur': HYBRID_FEATURES,
    'importance': hybrid_tuned_rf.feature_importances_
}).sort_values('importance', ascending=False).head(15)
sns.barplot(data=imp_df, x='importance', y='fitur', ax=axes[1], palette='viridis')
axes[1].set_title('Top-15 Feature Importance (Hybrid LSTM + RF)')

plt.tight_layout()
plt.savefig('rf_colab_plots.png', dpi=150)
plt.show()

In [ ]:
# 7. Simpan Log Eksperimen, Summary JSON & Download Model RF Terbaik
import json
from datetime import datetime

os.makedirs('models', exist_ok=True)
out_model = 'models/hybrid_rf_tuned.pkl'
joblib.dump(hybrid_tuned_rf, out_model)
results_df.to_csv('rf_benchmark_metrics.csv', index=False)

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
tuned_row = results_df[results_df['Model'] == 'Hybrid LSTM + RF (Tuned)'].iloc[0]
top5_str = '; '.join([f"{r['fitur']}:{round(float(r['importance']), 4)}" for _, r in imp_df.head(5).iterrows()])
colab_log = pd.DataFrame([{
    'run_id': run_id,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'git_commit': 'colab',
    'git_dirty': False,
    'stage': 'Hybrid_RF_Tuned',
    'environment': 'google_colab',
    'train_samples': len(y_train),
    'test_samples': len(y_test),
    'num_classes': len(classes),
    'hyperparameters': json.dumps(search.best_params_),
    'loss_akhir': '',
    'accuracy': tuned_row['Accuracy'],
    'precision_weighted': tuned_row['Precision (Weighted)'],
    'recall_weighted': tuned_row['Recall (Weighted)'],
    'f1_weighted': tuned_row['F1 (Weighted)'],
    'top_3_accuracy': tuned_row['Top-3 Accuracy'],
    'top_features': top5_str,
    'duration_sec': '',
    'catatan': 'RandomizedSearchCV TimeSeriesSplit (Colab)',
}])
colab_log.to_csv('colab_experiment_history.csv', index=False)

with open('colab_summary.json', 'w', encoding='utf-8') as f:
    json.dump({'run_id': run_id, 'best_params': search.best_params_, 'benchmark': results}, f, indent=2)

print('Model disimpan ke:', out_model)
print('Log eksperimen Colab disimpan ke: colab_experiment_history.csv & colab_summary.json')

# Jika di Google Colab, otomatis tawarkan download:
try:
    from google.colab import files
    for fname in [out_model, 'rf_benchmark_metrics.csv', 'colab_experiment_history.csv', 'colab_summary.json', 'rf_colab_plots.png']:
        files.download(fname)
except ImportError:
    pass